In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
import torch
from datasets import Dataset
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
    TrainerCallback
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

from trl import SFTTrainer
import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [2]:
df = pd.read_csv("final_combined_dataset.csv")

def format_example(row):
    text = f"""You are an expert MCQ solver.

Question: {row['question']}

Options:
a) {row['option_a']}
b) {row['option_b']}
c) {row['option_c']}
d) {row['option_d']}

INSTRUCTIONS:
- Choose exactly one: a, b, c, d
- If uncertain → idk
- Always provide reasoning

OUTPUT FORMAT (STRICT):
<answer>a/b/c/d/idk</answer>
<reasoning>2-3 line explanation</reasoning>

<answer>{row['final_answer']}</answer>
<reasoning>{row['reasoning']}</reasoning>
"""
    return {"text": text}

dataset = Dataset.from_list([format_example(row) for _, row in df.iterrows()])

In [3]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

model.gradient_checkpointing_enable()
model.config.use_cache = False


/home/sarmistha/miniconda3/envs/CIKM/lib/python3.9/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [4]:
lora_config = LoraConfig(
    r=8, 
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 5,046,272 || all params: 7,620,662,784 || trainable%: 0.06621828235983522


In [5]:
training_args = TrainingArguments(
    output_dir="./qwen_mcq_sft",
    per_device_train_batch_size=1,     
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-5,
    logging_steps=10,
    save_steps=200,
    fp16=True,
    optim="paged_adamw_32bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    report_to="none"
)

In [6]:
class TQDMProgressCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        self.pbar = tqdm(total=state.max_steps)

    def on_step_end(self, args, state, control, **kwargs):
        self.pbar.update(1)

    def on_train_end(self, args, state, control, **kwargs):
        self.pbar.close()

In [7]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    tokenizer=tokenizer,
    max_seq_length=512,   
    args=training_args,
    callbacks=[TQDMProgressCallback()]
)

Map:   0%|          | 0/3521 [00:00<?, ? examples/s]

/home/sarmistha/miniconda3/envs/CIKM/lib/python3.9/site-packages/accelerate/accelerator.py:469: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [8]:
trainer.train()

  0%|          | 0/1320 [00:00<?, ?it/s]

Step,Training Loss
10,2.928300
20,2.887200
30,2.888100
40,2.667400
50,2.543800
60,2.223800
70,2.003000
80,1.815000
90,1.531700
100,1.353100


/home/sarmistha/miniconda3/envs/CIKM/lib/python3.9/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/sarmistha/miniconda3/envs/CIKM/lib/python3.9/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/sarmistha/miniconda3/envs/CIKM/lib/python3.9/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/sarmistha/miniconda3/envs/CIKM/lib/python3.9/site-packages/huggingface

TrainOutput(global_step=1320, training_loss=0.8968116966160861, metrics={'train_runtime': 6821.8795, 'train_samples_per_second': 1.548, 'train_steps_per_second': 0.193, 'total_flos': 8.363092634917171e+16, 'train_loss': 0.8968116966160861, 'epoch': 2.999147969326896})

In [9]:
trainer.model.save_pretrained("qwen_mcq_sft_model")
tokenizer.save_pretrained("qwen_mcq_sft_model")

print("Training completed and model saved!")

/home/sarmistha/miniconda3/envs/CIKM/lib/python3.9/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Training completed and model saved!
